# StreamGuard — Colab runbook

**Phase 2, Stage 0: the BEFORE numbers.** Run top to bottom on a **T4 runtime**
(Runtime → Change runtime type → T4 GPU). One run installs everything, gets a YOLOX
checkpoint, exports it to ONNX, builds a TensorRT FP16 engine, and records the baseline
(`results/stage0_*.json`). The last cell prints everything to paste back.

Nothing in the GPU path has ever run before this notebook, so a failure is an expected,
useful result: paste the exact error. Do not work around it.

Set `RUN_M = True` below to also build the real baseline tier (YOLOX-m FP16 640) — adds
roughly 5–10 minutes of engine build and a second sweep.

## 1. Clone and install

In [ ]:
REPO_URL = "https://github.com/shruthib18122001-commits/StreamGuard.git"
RUN_M = False  # also build + sweep the ladder's level-0 tier (m / fp16 / 640)

!git clone $REPO_URL streamguard_repo
%cd streamguard_repo
!git log -1 --format='%H %s'

!pip install -e ".[viz]" -q
# GPU libraries are not in pyproject.toml (modules import them lazily so the package
# installs and CI runs without them). Pinned to the CUDA 12 builds: an unsuffixed
# `pip install tensorrt` can resolve to a CUDA 13 build that will not match Colab's runtime.
!pip install -q cupy-cuda12x tensorrt-cu12 pynvml

## 2. Confirm GPU, driver, CUDA and TensorRT versions

In [ ]:
!nvidia-smi
from streamguard.metrics.device import capture_device_info

info = capture_device_info("cuda:0")
print(info)
assert info.driver_version is not None, "no driver visible -- not a GPU runtime"
assert info.tensorrt_version is not None, "tensorrt did not import"
assert info.cuda_runtime_version is not None, "cupy could not query CUDA"
if "T4" not in (info.gpu_name or ""):
    print(f"NOTE: GPU is {info.gpu_name!r}, not a T4 -- results are not comparable with the T4 baseline")

## 3. Export YOLOX to ONNX (dynamic batch, NCHW, undecoded)

Uses YOLOX's checkpoints from its GitHub release and `scripts/export_yolox_onnx.py` (see its
docstring for why it differs from YOLOX's own exporter). The pre-exported `.onnx` files
YOLOX publishes are static batch-1 and cannot be used for batching.

In [ ]:
!git clone -q --depth 1 https://github.com/Megvii-BaseDetection/YOLOX /content/YOLOX
!pip install -q loguru thop tabulate pycocotools onnx psutil
!mkdir -p /content/ckpt /content/onnx_models

REL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0"
models = ["tiny"] + (["m"] if RUN_M else [])
for m in models:
    !wget -q -nc -O /content/ckpt/yolox_{m}.pth {REL}/yolox_{m}.pth
    !python scripts/export_yolox_onnx.py --yolox-dir /content/YOLOX --name yolox-{m} --ckpt /content/ckpt/yolox_{m}.pth --output /content/onnx_models/yolox_{m}.onnx

## 4. Build TensorRT FP16 engines

`tiny / fp16 / 416` is the Stage 0 baseline engine. The build script now declares an NCHW
profile and refuses a static-batch ONNX with a clear message.

In [ ]:
import os
os.makedirs("engines", exist_ok=True)
!python scripts/build_engine.py --onnx /content/onnx_models/yolox_tiny.onnx --output-dir engines --tier tiny --precision fp16 --input-size 416
if RUN_M:
    !python scripts/build_engine.py --onnx /content/onnx_models/yolox_m.onnx --output-dir engines --tier m --precision fp16 --input-size 640
!ls -la engines

## 5. A real video

Intel's public sample clip (Intel IoT DevKit sample-videos). Its resolution is recorded in the results
JSON (`frame_shape_hwc`). To use your own footage instead, put `.mp4` files in `/content/sample_videos`
and skip the download.

In [ ]:
!mkdir -p /content/sample_videos
!wget -q -nc -O /content/sample_videos/clip.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4
import cv2
cap = cv2.VideoCapture("/content/sample_videos/clip.mp4")
print("opened:", cap.isOpened(), "frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
      "size:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
      "fps:", cap.get(cv2.CAP_PROP_FPS))
assert cap.isOpened()

## 6. Baseline: per-stage latency, bytes moved, NVML, NVDEC attempt

Sanity-checks the engine on a real frame first (detections must appear, otherwise the layout or
export is wrong and the timings mean nothing), then measures batch 1 and 8 with CUDA events,
runs the single-stream pipeline, and makes one time-boxed NVDEC attempt in a subprocess.

In [ ]:
!python scripts/run_baseline.py --engine-dir engines --tier tiny --precision fp16 --input-size 416 \
    --video /content/sample_videos/clip.mp4 --output results/stage0_baseline_tiny_fp16_416.json

## 7. Max sustainable stream count at the baseline tier

Ramps stream count with the existing multi-stream pipeline (CPU decode, CPU preprocess, host path)
until p99 exceeds 33 ms. `tiny / fp16 / 416` is the tier whose engine we built; level 0
(`m / fp16 / 640`) runs only if `RUN_M`.

In [ ]:
!python scripts/run_stream_count_sweep.py --backend tensorrt --engine-dir engines --source-dir /content/sample_videos \
    --model-tier tiny --precision fp16 --input-size 416 --max-frames-per-stream 150 \
    --output results/sweep_tiny_fp16_416.csv --json-output results/stage0_sweep_tiny_fp16_416.json
if RUN_M:
    !python scripts/run_stream_count_sweep.py --backend tensorrt --engine-dir engines --source-dir /content/sample_videos \
        --tier-level 0 --max-frames-per-stream 150 \
        --output results/sweep_m_fp16_640.csv --json-output results/stage0_sweep_m_fp16_640.json

## 8. Paste this back

Prints every Stage 0 JSON between markers. Copy everything from `=====BEGIN` to `=====END` and paste it
into the chat (also paste any cell that errored, in full). `results/` is git-ignored on purpose.

In [ ]:
import glob
print("=====BEGIN STAGE0")
for p in sorted(glob.glob("results/stage0_*.json")):
    print(f"--- {p}")
    print(open(p).read())
print("=====END STAGE0")